In [0]:
employee_df = spark.read.csv(
    path="/Volumes/quickstart_catalog/quickstart_schema/sandbox/dataset/employee.csv",
    header=True,
    inferSchema=True,
    sep="|",
    quote="'"
)

employee_df.display()

In [0]:
from pyspark.sql.functions import col
employee_df.select(col("id"), col("name").alias("full_name")).display()

In [0]:
columns_to_exclude = ["dob"]
columns_to_include = []
for column in employee_df.columns:
    if column not in columns_to_exclude:
        columns_to_include.append(column)
employee_df.select(*columns_to_include).display()

In [0]:
columns_to_exclude = ["dob"]
columns_to_include = list(
    filter(lambda column: column not in columns_to_exclude, employee_df.columns)
)
employee_df.select(*columns_to_include).display()

In [0]:
# [e for e in L if e%2==0]
columns_to_exclude = ["dob", "id"]
columns_to_include = [
    column for column in employee_df.columns if column not in columns_to_exclude
]

employee_df.select(*columns_to_include).display()

In [0]:
# display(employee_df.filter(col("gen")=="M"))
employee_df.filter(col("gen")=="M").select("name").display()

In [0]:
employee_df.filter((col("gen")=="M") & (col('exp') > 2)).display()

In [0]:
employee_df.filter(col("company")=="cisco").select("name").display()

In [0]:
# employee_df.filter((col("desig")=="Team Lead") | (col("desig")=="Developer")).display()

employee_df.filter(col("desig").isin(["Team Lead", "Developer"])).display()

In [0]:
# employee_df.groupBy("gen").count().sort(col("count"), desc()).display()
employee_df.groupBy("gen").count().orderBy(col("count"), ascending=False).display()

In [0]:
from pyspark.sql.functions import when

employee_df.withColumn(
    "exp_level",
    when(col("exp") >= 10, "Senior")
    .when(col("exp") >= 5, "Mid lvl")
    .when(col("exp") >= 0, "Junior")
    .otherwise("Invalid Experience"),
).select("name", "exp", "exp_level").display()

In [0]:
employee_df.selectExpr(
    "*",
    """
    CASE
        WHEN exp >= 10 THEN 'Senior'
        WHEN exp >= 5 THEN 'Mid Level'
        WHEN exp >= 0 THEN 'Junior'
        ELSE 'Invalid Exp'
    END AS exp_level
    """,
).display()

In [0]:
employee_df.createOrReplaceTempView("employee_vw")

In [0]:
%sql
SELECT
  name,
  exp,
  CASE
    WHEN exp >= 10 THEN 'Senior'
    WHEN exp >= 5 THEN 'Mid Level'
    WHEN exp >= 0 THEN 'Junior'
    ELSE 'Invalid Exp'
  END AS exp_level
from
  employee_vw;